# A.- HEADER - LEENDO BYTES

## 1.- HEADER
1. Header fijo (256 bytes)
2. Header por canal (256 bytes × nº canales)


### 1.1 HEADER FIJO (PRIMEROS 256 BYTES)


| Bytes   | Tamaño | Significado        |
| ------- | ------ | ------------------ |
| 0–7     | 8      | versión            |
| 8–87    | 80     | patient ID         |
| 88–167  | 80     | recording ID       |
| 168–175 | 8      | fecha              |
| 176–183 | 8      | hora               |
| 184–191 | 8      | tamaño header      |
| 192–235 | 44     | reservado          |
| 236–243 | 8      | nº data records    |
| 244–251 | 8      | duración de record |
| 252–255 | 4      | nº canales         |


#### Lectura manual del header (nivel binario)

Vamos a leer directamente los primeros bytes del archivo `.bdf` para entender su estructura interna.

Esto nos permite verificar cómo está organizado el archivo según el estándar EDF/BDF.

In [281]:
from pathlib import Path
from typing import BinaryIO

file_path: Path = Path("../dataset/raw/bdf/s32.bdf")

with open(file_path, "rb") as f:
    header: bytes = f.read(520)

print(f"Tamaño del header leído: {len(header)} bytes")

Tamaño del header leído: 520 bytes


In [282]:
from typing import Tuple

def read_str(data: bytes, start: int, length: int) -> str:
    return data[start : start + length].decode("ascii", errors="ignore").strip()

In [283]:
version: str = read_str(header, 0, 8)
patient_id: str = read_str(header, 8, 80)
recording_id: str = read_str(header, 88, 80)
start_date: str = read_str(header, 168, 8)
start_time: str = read_str(header, 176, 8)
header_bytes: str = read_str(header, 184, 8)
reserved: str = read_str(header, 192, 44)
num_records: str = read_str(header, 236, 8)
record_duration: str = read_str(header, 244, 8)
num_channels: str = read_str(header, 252, 4)

print("Versión:", version)
print("Paciente:", patient_id)
print("Recording:", recording_id)
print("Fecha:", start_date)
print("Hora:", start_time)
print("Header bytes:", header_bytes)
print("Reservado:", reserved)
print("N° records:", num_records)
print("Duración record:", record_duration)
print("N° canales:", num_channels)

header_bytes: int = int(header_bytes)
num_channels:int = int(num_channels)

Versión: BIOSEMI
Paciente: A19
Recording: unige
Fecha: 04.08.10
Hora: 11.23.36
Header bytes: 12800
Reservado: 24BIT
N° records: 3524
Duración record: 1
N° canales: 49


### 1.2 HEADER POR CANAL (LO MÁS INTERESANTE)
- cada canal tiene 256 bytes

| Campo              | Tamaño |
| ------------------ | ------ |
| label              | 16     |
| transducer         | 80     |
| physical dimension | 8      |
| physical min       | 8      |
| physical max       | 8      |
| digital min        | 8      |
| digital max        | 8      |
| prefiltering       | 80     |
| samples per record | 8      |
| reserved           | 32     |


#### Lectura correcta del header completo

El archivo `.bdf` tiene:

1. Header general: 256 bytes.
2. Header de señales/canales: 256 bytes × número de canales.

Pero los datos de los canales no están almacenados como:

```text
canal 1 completo, canal 2 completo, canal 3 completo...
```
sino por bloques:

- labels de todos los canales
- transducers de todos los canales
- unidades físicas de todos los canales
- mínimos físicos de todos los canales
- máximos físicos de todos los canales
- ....
- ....
- ....
- reserved de todos los canales


In [284]:
with open(file_path, "rb") as file:
    full_header: bytes = file.read(header_bytes)

print(f"Bytes leídos del header completo: {len(full_header)}")

Bytes leídos del header completo: 12800


#### Extraer nombres de canales

Después de los primeros 256 bytes, vienen los nombres de todos los canales.

Cada nombre ocupa 16 bytes.

Como tenemos 48 canales:

```text
48 × 16 = 768 bytes

In [285]:
labels_start: int = 256
label_size: int = 16

channel_labels: list[str] = [
    read_str(full_header, labels_start + channel_index * label_size, label_size)
    for channel_index in range(int(num_channels))
]

for index, label in enumerate(channel_labels, start=1):
    print(f"{index:02d}. {label}")

01. Fp1
02. AF3
03. F3
04. F7
05. FC5
06. FC1
07. C3
08. T7
09. CP5
10. CP1
11. P3
12. P7
13. PO3
14. O1
15. Oz
16. Pz
17. Fp2
18. AF4
19. Fz
20. F4
21. F8
22. FC6
23. FC2
24. Cz
25. C4
26. T8
27. CP6
28. CP2
29. P4
30. P8
31. PO4
32. O2
33. EXG1
34. EXG2
35. EXG3
36. EXG4
37. EXG5
38. EXG6
39. EXG7
40. EXG8
41. GSR1
42. GSR2
43. Erg1
44. Erg2
45. Resp
46. Plet
47. Temp
48. 
49. 


In [286]:
transducer_size: int = 80
physical_dimension_size: int = 8

transducers_start: int = labels_start + (num_channels * label_size)
physical_dimensions_start: int = transducers_start + (num_channels * transducer_size)

physical_dimensions: list[str] = [
    read_str(
        full_header,
        physical_dimensions_start + channel_index * physical_dimension_size,
        physical_dimension_size,
    )
    for channel_index in range(num_channels)
]

for label, unit in zip(channel_labels, physical_dimensions):
    print(f"{label}: {unit}")

Fp1: uV
AF3: uV
F3: uV
F7: uV
FC5: uV
FC1: uV
C3: uV
T7: uV
CP5: uV
CP1: uV
P3: uV
P7: uV
PO3: uV
O1: uV
Oz: uV
Pz: uV
Fp2: uV
AF4: uV
Fz: uV
F4: uV
F8: uV
FC6: uV
FC2: uV
Cz: uV
C4: uV
T8: uV
CP6: uV
CP2: uV
P4: uV
P8: uV
PO4: uV
O2: uV
EXG1: uV
EXG2: uV
EXG3: uV
EXG4: uV
EXG5: uV
EXG6: uV
EXG7: uV
EXG8: uV
GSR1: Ohm
GSR2: Ohm
Erg1: uV
Erg2: uV
Resp: uV
Plet: uV
Temp: Celsius
: Boolean
: Boolean


___

#### Lectura completa de los headers por canal

En EDF/BDF, después del header fijo de 256 bytes, aparecen los campos de todos los canales agrupados por tipo de campo.

El orden es:

1. `label` → 16 bytes por canal
2. `transducer` → 80 bytes por canal
3. `physical_dimension` → 8 bytes por canal
4. `physical_min` → 8 bytes por canal
5. `physical_max` → 8 bytes por canal
6. `digital_min` → 8 bytes por canal
7. `digital_max` → 8 bytes por canal
8. `prefiltering` → 80 bytes por canal
9. `samples_per_record` → 8 bytes por canal
10. `reserved` → 32 bytes por canal

Como tenemos 48 canales, cada bloque tiene `tamaño_del_campo × 48` bytes.

#### Definir estructura de campos del header por canal

Ahora definimos el nombre y tamaño de cada campo. Luego calculamos automáticamente dónde empieza cada bloque dentro del header completo.

In [287]:
from typing import Dict, List, Tuple

signal_header_start: int = 256

field_sizes: List[Tuple[str, int]] = [
    ("label", 16),
    ("transducer", 80),
    ("physical_dimension", 8),
    ("physical_min", 8),
    ("physical_max", 8),
    ("digital_min", 8),
    ("digital_max", 8),
    ("prefiltering", 80),
    ("samples_per_record", 8),
    ("reserved", 32),
]

field_starts: Dict[str, int] = {}

current_start: int = signal_header_start

for field_name, field_size in field_sizes:
    field_starts[field_name] = current_start
    current_start += field_size * num_channels

for field_name, start in field_starts.items():
    print(f"{field_name}: empieza en byte {start}")

label: empieza en byte 256
transducer: empieza en byte 1040
physical_dimension: empieza en byte 4960
physical_min: empieza en byte 5352
physical_max: empieza en byte 5744
digital_min: empieza en byte 6136
digital_max: empieza en byte 6528
prefiltering: empieza en byte 6920
samples_per_record: empieza en byte 10840
reserved: empieza en byte 11232


#### Extraer todos los campos para todos los canales

Aquí construiremos una tabla donde cada fila representa un canal y cada columna representa un campo del header.

In [288]:
from typing import List, Dict, Any
import pandas as pd

channel_rows: List[Dict[str, Any]] = []

for channel_index in range(num_channels):
    row: Dict[str, Any] = {"channel_index": channel_index + 1}

    for field_name, field_size in field_sizes:
        start: int = field_starts[field_name] + channel_index * field_size
        value: str = read_str(full_header, start, field_size)
        row[field_name] = value

    channel_rows.append(row)

header_df: pd.DataFrame = pd.DataFrame(channel_rows)

header_df

,channel_index,label,transducer,physical_dimension,physical_min,physical_max,digital_min,digital_max,prefiltering,samples_per_record,reserved
0,1,Fp1,Active Electrode,uV,-262144,262143,-8388608,8388607,HP: DC; LP: 104 Hz,512,MON
1,2,AF3,Active Electrode,uV,-262144,262143,-8388608,8388607,HP: DC; LP: 104 Hz,512,MON
2,3,F3,Active Electrode,uV,-262144,262143,-8388608,8388607,HP: DC; LP: 104 Hz,512,MON
3,4,F7,Active Electrode,uV,-262144,262143,-8388608,8388607,HP: DC; LP: 104 Hz,512,MON
4,5,FC5,Active Electrode,uV,-262144,262143,-8388608,8388607,HP: DC; LP: 104 Hz,512,MON
5,6,FC1,Active Electrode,uV,-262144,262143,-8388608,8388607,HP: DC; LP: 104 Hz,512,MON
6,7,C3,Active Electrode,uV,-262144,262143,-8388608,8388607,HP: DC; LP: 104 Hz,512,MON
7,8,T7,Active Electrode,uV,-262144,262143,-8388608,8388607,HP: DC; LP: 104 Hz,512,MON
8,9,CP5,Active Electrode,uV,-262144,262143,-8388608,8388607,HP: DC; LP: 104 Hz,512,MON
9,10,CP1,Active Electrode,uV,-262144,262143,-8388608,8388607,HP: DC; LP: 104 Hz,512,MON


#### Imprimir la tabla completa en texto

Si la tabla se ve cortada en Jupyter, podemos imprimir cada canal de forma ordenada.

In [289]:
for _, row in header_df.iterrows():
    print("=" * 80)
    print(f"Canal {row['channel_index']:02d}: {row['label']}")
    print(f"Transducer: {row['transducer']}")
    print(f"Physical dimension: {row['physical_dimension']}")
    print(f"Physical min: {row['physical_min']}")
    print(f"Physical max: {row['physical_max']}")
    print(f"Digital min: {row['digital_min']}")
    print(f"Digital max: {row['digital_max']}")
    print(f"Prefiltering: {row['prefiltering']}")
    print(f"Samples per record: {row['samples_per_record']}")
    print(f"Reserved: {row['reserved']}")

Canal 01: Fp1
Transducer: Active Electrode
Physical dimension: uV
Physical min: -262144
Physical max: 262143
Digital min: -8388608
Digital max: 8388607
Prefiltering: HP: DC; LP: 104 Hz
Samples per record: 512
Reserved: MON
Canal 02: AF3
Transducer: Active Electrode
Physical dimension: uV
Physical min: -262144
Physical max: 262143
Digital min: -8388608
Digital max: 8388607
Prefiltering: HP: DC; LP: 104 Hz
Samples per record: 512
Reserved: MON
Canal 03: F3
Transducer: Active Electrode
Physical dimension: uV
Physical min: -262144
Physical max: 262143
Digital min: -8388608
Digital max: 8388607
Prefiltering: HP: DC; LP: 104 Hz
Samples per record: 512
Reserved: MON
Canal 04: F7
Transducer: Active Electrode
Physical dimension: uV
Physical min: -262144
Physical max: 262143
Digital min: -8388608
Digital max: 8388607
Prefiltering: HP: DC; LP: 104 Hz
Samples per record: 512
Reserved: MON
Canal 05: FC5
Transducer: Active Electrode
Physical dimension: uV
Physical min: -262144
Physical max: 262143
D

### 1.3 Interpretación del header de canales

El archivo `s01.bdf` contiene 48 canales:

- Canales 1–32: EEG.
- Canales 33–40: EXG, señales auxiliares registradas en microvoltios.
- Canales 41–47: señales fisiológicas periféricas.
- Canal 48: `Status`, usado para triggers/eventos del experimento.

El header también indica la unidad física, el rango físico, el rango digital y el filtrado aplicado o declarado para cada canal.

Los campos `digital_min` y `digital_max` representan el rango de valores enteros almacenados en el archivo.  
Los campos `physical_min` y `physical_max` indican cómo esos valores digitales deben convertirse a unidades físicas como `uV`, `nS` o `Celsius`.

# B.- HEADER - USANDO MNE

## 1.- HEADER
1. Header fijo (256 bytes)
2. Header por canal (256 bytes × nº canales)

### 1.1 HEADER FIJO (PRIMEROS 256 BYTES)

In [290]:
from pathlib import Path
from typing import Any, Dict, List

import mne
import pandas as pd
from mne.io import BaseRaw

file_path: Path = Path("../dataset/s01.bdf")

raw: BaseRaw = mne.io.read_raw_bdf(
    input_fname=file_path,
    preload=False,
    verbose=True,
)

info: Dict[str, Any] = raw.info
info

FileNotFoundError: File does not exist: "/home/russell/ssd/code/Topicos_Ciencia_Datos/Visual_Analytic_DEAP/notebooks/../dataset/s01.bdf"

In [ ]:
from typing import Any, Dict


def get_mne_fixed_header(raw: BaseRaw) -> Dict[str, Any]:
    """
    Extrae información general del archivo BDF usando la estructura Raw de MNE.

    Nota:
    MNE no expone el header fijo EDF/BDF exactamente con los mismos campos
    del archivo binario. Esta función reconstruye la información equivalente
    disponible desde raw.info y atributos del objeto Raw.
    """
    fixed_header: Dict[str, Any] = {
        "file_name": Path(raw.filenames[0]).name if raw.filenames else None,
        "meas_date": raw.info.get("meas_date"),
        "n_channels": raw.info.get("nchan"),
        "sampling_frequency_hz": raw.info.get("sfreq"),
        "n_times": raw.n_times,
        "duration_seconds": raw.n_times / float(raw.info["sfreq"]),
        "duration_minutes": (raw.n_times / float(raw.info["sfreq"])) / 60.0,
        "channel_names": raw.ch_names,
        "bad_channels": raw.info.get("bads"),
        "highpass": raw.info.get("highpass"),
        "lowpass": raw.info.get("lowpass"),
    }

    return fixed_header


mne_fixed_header: Dict[str, Any] = get_mne_fixed_header(raw)

for key, value in mne_fixed_header.items():
    print(f"{key}: {value}")

Data leendo bytes:
- Versión: BIOSEMI
- Paciente: s01
- Recording: 
- Fecha: 01.07.10
- Hora: 10.00.16
- Header bytes: 12544
- Reservado: 24BIT
- N° records: 3869
- Duración record: 1
- N° canales: 48

___

### 1.2 HEADER POR CANAL (LO MÁS INTERESANTE)
- cada canal tiene 256 bytes
- Intentar ver información interna que MNE leyó del BDF

In [ ]:
from typing import Any, Dict

raw_extra: Dict[str, Any] = raw._raw_extras[0]

print("Claves disponibles en raw._raw_extras[0]:")
for key in raw_extra.keys():
    print(key)

In [ ]:
from typing import Any, Dict, List

import pandas as pd


def build_mne_channel_header_df(raw: BaseRaw) -> pd.DataFrame:
    """
    Construye una tabla por canal usando la información disponible en MNE.

    Esta tabla no es una copia literal del header BDF crudo, pero sí representa
    la interpretación que MNE hace de cada canal.
    """
    rows: List[Dict[str, Any]] = []

    for channel_index, channel_info in enumerate(raw.info["chs"], start=1):
        row: Dict[str, Any] = {
            "channel_index": channel_index,
            "label": channel_info.get("ch_name"),
            "kind": channel_info.get("kind"),
            "unit": channel_info.get("unit"),
            "unit_mul": channel_info.get("unit_mul"),
            "coil_type": channel_info.get("coil_type"),
            "coord_frame": channel_info.get("coord_frame"),
            "cal": channel_info.get("cal"),
            "range": channel_info.get("range"),
            "loc": channel_info.get("loc"),
        }

        rows.append(row)

    return pd.DataFrame(rows)


mne_channel_df: pd.DataFrame = build_mne_channel_header_df(raw)

mne_channel_df

In [ ]:
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd


def get_raw_extra_value(
    raw_extra: Dict[str, Any],
    key: str,
    channel_index: int,
) -> Optional[Any]:
    """
    Intenta extraer un valor por canal desde raw._raw_extras[0].

    Si la clave no existe o no tiene estructura por canal, retorna None.
    """
    if key not in raw_extra:
        return None

    value: Any = raw_extra[key]

    try:
        if isinstance(value, (list, tuple, np.ndarray)) and len(value) > channel_index:
            return value[channel_index]
    except TypeError:
        return None

    return None

print(data.shape)
def build_bdf_like_header_from_mne(raw: BaseRaw) -> pd.DataFrame:
    """
    Construye una tabla con nombres similares al header BDF leído manualmente.

    Importante:
    Algunas columnas pueden quedar como None porque MNE no expone todos los
    campos originales del header BDF de manera pública.
    """
    raw_extra: Dict[str, Any] = raw._raw_extras[0]
    rows: List[Dict[str, Any]] = []

    for zero_based_index, channel_name in enumerate(raw.ch_names):
        row: Dict[str, Any] = {
            "channel_index": zero_based_index + 1,
            "label": channel_name,
            "transducer": get_raw_extra_value(
                raw_extra, "transducer", zero_based_index
            ),
            "physical_dimension": get_raw_extra_value(
                raw_extra, "units", zero_based_index
            ),
            "physical_min": get_raw_extra_value(
                raw_extra, "physical_min", zero_based_index
            ),
            "physical_max": get_raw_extra_value(
                raw_extra, "physical_max", zero_based_index
            ),
            "digital_min": get_raw_extra_value(
                raw_extra, "digital_min", zero_based_index
            ),
            "digital_max": get_raw_extra_value(
                raw_extra, "digital_max", zero_based_index
            ),
            "prefiltering": get_raw_extra_value(
                raw_extra, "prefiltering", zero_based_index
            ),
            "samples_per_record": get_raw_extra_value(
                raw_extra, "n_samps", zero_based_index
            ),
            "reserved": get_raw_extra_value(raw_extra, "reserved", zero_based_index),
        }

        rows.append(row)

    return pd.DataFrame(rows)


mne_bdf_like_df: pd.DataFrame = build_bdf_like_header_from_mne(raw)

mne_bdf_like_df

In [ ]:
raw_extra: Dict[str, Any] = raw._raw_extras[0]

for key, value in raw_extra.items():
    print("=" * 80)
    print("KEY:", key)
    print("TYPE:", type(value))
    print("VALUE:", value if not isinstance(value, (list, tuple)) else value[:5])

# DATA RECORD + manual

Celda 1: funciones auxiliares

In [ ]:
from pathlib import Path
from typing import List

import mne
import numpy as np
import pandas as pd
from mne.io import BaseRaw


def decode_bdf_24bit_sample(sample_bytes: bytes) -> int:
    """
    Convierte 3 bytes BDF little-endian en un entero signed de 24 bits.
    """
    if len(sample_bytes) != 3:
        raise ValueError("Una muestra BDF debe tener exactamente 3 bytes.")

    value: int = int.from_bytes(sample_bytes, byteorder="little", signed=False)

    # Sign extension para entero signed de 24 bits
    if value >= 2**23:
        value -= 2**24

    return value


def digital_to_physical(
    digital_value: int,
    digital_min: int,
    digital_max: int,
    physical_min: float,
    physical_max: float,
) -> float:
    """
    Convierte un valor digital crudo a valor físico usando la escala EDF/BDF.
    """
    physical_value: float = physical_min + (
        (digital_value - digital_min) * (physical_max - physical_min)
    ) / (digital_max - digital_min)

    return physical_value

Celda 2: leer manualmente las primeras 20 muestras de un canal

In [ ]:
file_path: Path = Path("../dataset/s01.bdf")

channel_index: int = 0  # Fp1
record_index: int = 0  # primer data record
num_samples_to_read: int = 20

header_bytes_int: int = int(header_bytes)

samples_per_record: int = int(header_df.loc[channel_index, "samples_per_record"])
num_channels_int: int = int(num_channels)

bytes_per_sample: int = 3
bytes_per_channel_per_record: int = samples_per_record * bytes_per_sample
bytes_per_data_record: int = num_channels_int * bytes_per_channel_per_record

channel_offset_in_record: int = channel_index * bytes_per_channel_per_record

start_byte: int = (
    header_bytes_int + record_index * bytes_per_data_record + channel_offset_in_record
)

manual_digital_values: List[int] = []

with open(file_path, "rb") as file:
    file.seek(start_byte)

    for _ in range(num_samples_to_read):
        sample_bytes: bytes = file.read(bytes_per_sample)
        digital_value: int = decode_bdf_24bit_sample(sample_bytes)
        manual_digital_values.append(digital_value)

print("Valores digitales manuales:")
print(manual_digital_values)

Celda 3: convertir esos valores digitales a físicos

In [ ]:
physical_min: float = float(header_df.loc[channel_index, "physical_min"])
physical_max: float = float(header_df.loc[channel_index, "physical_max"])
digital_min: int = int(header_df.loc[channel_index, "digital_min"])
digital_max: int = int(header_df.loc[channel_index, "digital_max"])

manual_physical_values_uv: List[float] = [
    digital_to_physical(
        digital_value=digital_value,
        digital_min=digital_min,
        digital_max=digital_max,
        physical_min=physical_min,
        physical_max=physical_max,
    )
    for digital_value in manual_digital_values
]

print("Valores físicos manuales en uV:")
print(manual_physical_values_uv)

Celda 4: obtener los mismos valores usando MNE

In [ ]:
raw: BaseRaw = mne.io.read_raw_bdf(
    input_fname=file_path,
    preload=False,
    verbose=False,
)

mne_values_v: np.ndarray = raw.get_data(
    picks=[channel_index],
    start=0,
    stop=num_samples_to_read,
)[0]

mne_values_uv: np.ndarray = mne_values_v * 1_000_000

print("Valores MNE en uV:")
print(mne_values_uv)

Celda 5: comparación en un DataFrame

In [ ]:
comparison_df: pd.DataFrame = pd.DataFrame(
    {
        "sample_index": list(range(num_samples_to_read)),
        "manual_digital": manual_digital_values,
        "manual_physical_uV": manual_physical_values_uv,
        "mne_uV": mne_values_uv,
    }
)

comparison_df["difference_uV"] = (
    comparison_df["manual_physical_uV"] - comparison_df["mne_uV"]
)

comparison_df

Qué deberías esperar

Si todo está correcto, la diferencia debería ser muy pequeña, cercana a cero.

Puede haber diferencias mínimas por redondeo numérico, pero no debería haber una diferencia grande.

Conceptualmente estarías validando esto:

bytes crudos del archivo
→ entero digital signed de 24 bits
→ conversión digital a físico
→ mismo valor que MNE

Y eso demostraría que ya entiendes cómo MNE reconstruye internamente la señal desde el .bdf

# DATA RECORD + mne


In [ ]:
data = raw.get_data()
print(data.shape)

Acceder a canales especificos

In [ ]:
fp1 = data[0]
print(fp1[:10])

In [ ]:
status = data[47]
print(status[:100])